# 2.1 — Cross-Sectional Normative Model (BLR)

Fits a Bayesian Linear Regression normative model with B-spline age basis across all structural MRI ROIs.

**Input:** `normative_train.csv` produced by `03_train_test_split.ipynb`  
**Outputs:** Z-scores, global metrics, site-specific metrics, ROI mapping table, colorbar figure  
**Environment:** `normmodel310` (PCNToolkit ≥ 0.29)

Pipeline steps:
1. Load data & map sites
2. Feature / covariate prep + one-hot site encoding
3. 80/20 internal train/test split
4. Save base covariate & response files
5. Define B-spline basis (age)
6. ROI loop: estimate BLR → global eval → site eval
7. Save aggregated metrics & Z-scores
8. ROI name mapping + summary stats
9. Metric colorbars figure

In [1]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Set these paths before running.

# INPUT_CSV  = 'data/splits/normative_train2.csv'   # output of 03_train_test_split.ipynb
# OUTPUT_DIR = 'outputs/normative_model'             # all model outputs written here


# B-spline age range (years) — must match study population
AGE_XMIN, AGE_XMAX, AGE_NKNOTS = 7, 18, 3

# Internal train/test split within normative_train2
INTERNAL_TEST_SIZE = 0.20
RANDOM_STATE       = 42

# Minimum site N for site-level evaluation
MIN_SITE_N = 10

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from pcntoolkit.normative import estimate, evaluate
from pcntoolkit.util.utils import create_bspline_basis

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'ROI_models'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

## Step 1 — Load data & map sites

In [3]:
# ABCD HASH → integer site labels (30 sites)
SITE_MAPPING = {
    'HASH4d1ed7b1_9': 1,  'HASH3935c89e_16': 2,  'HASHd7cb4c6d_10': 3,
    'HASHe3ce02d3_10': 4, 'HASH7911780b_15': 5,  'HASH3ce956bd_18': 6,
    'HASHa3e45734_18': 7, 'HASH4036a433_21': 8,  'HASHdb2589d4_17': 9,
    'HASH1314a204_2': 10, 'HASHfeb7e81a_4': 11,  'HASHc3bf3d9c_13': 12,
    'HASH69f406fa_13': 13,'HASH96a0c182_6': 14,  'HASHe4f6957a_12': 15,
    'HASH4b0b8b05_4': 16, 'HASH11ad4ed5_14': 17, 'HASH7f91147d_14': 18,
    'HASH65b39280_7': 19, 'HASH311170b9_5': 20,  'HASHe76e6d72_21': 21,
    'HASHb640a1b8_21': 22,'HASH03db707f_11': 23, 'HASHd422be27_20': 24,
    'HASHc9398971_20': 25,'HASH5b0cf1bb_3': 26,  'HASH5b2fcf80_8': 27,
    'HASH5ac2b20b_19': 28,'HASHc1b3365b_13': 29, 'HASH6b4422a7_1': 30,
}
N_SITES = max(SITE_MAPPING.values())

all_data = pd.read_csv(INPUT_CSV).set_index('ID-wave')
all_data['site_mri'] = all_data['site_mri'].map(SITE_MAPPING).astype('Int64')

print(f'Loaded: {len(all_data)} rows')
print(f'Sites represented: {all_data["site_mri"].nunique()} / {N_SITES}')

Loaded: 6373 rows
Sites represented: 30 / 30


## Step 2 — Feature / covariate prep

In [4]:
roi_ids = all_data.columns[all_data.columns.str.contains('mr_y_smri__')].tolist()
print(f'ROIs: {len(roi_ids)}')

features   = all_data[roi_ids]
covariates = all_data[['ab_g_dyn__visit_age', 'ab_g_stc__cohort_sex', 'site_mri']]
covariates = pd.get_dummies(covariates, columns=['site_mri'], dtype=int)

ROIs: 85


## Step 3 — 80/20 internal train/test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    covariates, features,
    test_size=INTERNAL_TEST_SIZE,
    random_state=RANDOM_STATE,
)
for df in [X_train, X_test, y_train, y_test]:
    df.reset_index(drop=True, inplace=True)

# Per-site test indices (used for site-level evaluation)
sites      = [X_test.index[X_test[f'site_mri_{i}'] == 1].tolist() for i in range(1, N_SITES+1)]
site_names = [f'site_{i}' for i in range(1, N_SITES+1)]

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

Train: 5098 | Test: 1275


## Step 4 — Save base covariate & response files

In [6]:
y_train.to_csv(os.path.join(OUTPUT_DIR, 'resp_tr.txt'), sep='\t', header=False, index=False)
y_test.to_csv( os.path.join(OUTPUT_DIR, 'resp_te.txt'), sep='\t', header=False, index=False)
X_train.to_csv(os.path.join(OUTPUT_DIR, 'cov_tr.txt'),  sep='\t', header=False, index=False)
X_test.to_csv( os.path.join(OUTPUT_DIR, 'cov_te.txt'),  sep='\t', header=False, index=False)

for c in y_train.columns:
    y_train[c].to_csv(os.path.join(OUTPUT_DIR, f'resp_tr_{c}.txt'), header=False, index=False)
    y_test[c].to_csv( os.path.join(OUTPUT_DIR, f'resp_te_{c}.txt'), header=False, index=False)

print('Base files saved.')

Base files saved.


## Step 5 — B-spline basis (age)

In [7]:
B = create_bspline_basis(AGE_XMIN, AGE_XMAX, nknots=AGE_NKNOTS)
print(f'B-spline basis: age [{AGE_XMIN}, {AGE_XMAX}], nknots={AGE_NKNOTS}')

B-spline basis: age [7, 18], nknots=3


## Step 6 — ROI modelling loop

In [8]:
blr_metrics        = pd.DataFrame(columns=['ROI','MSLL','EXPV','SMSE','RMSE','Rho','pRho'])
blr_globmetrics    = pd.DataFrame(columns=['ROI','site','MSLL','EXPV','SMSE','RMSE','Rho','pRho'])
blr_site_metrics   = pd.DataFrame(columns=['ROI','site','MSLL','EXPV','SMSE','RMSE','Rho','pRho'])
metrics_yhat_s2_Z  = pd.DataFrame(columns=['ROI','yhat_te','s2_te','Z'])

for roi in roi_ids:
    print(f'Running ROI: {roi}')
    roi_dir = os.path.join(OUTPUT_DIR, roi)
    os.makedirs(roi_dir, exist_ok=True)
    os.makedirs(os.path.join(roi_dir, 'blr'), exist_ok=True)

    # Load base covariates, add intercept
    X_tr = np.loadtxt(os.path.join(OUTPUT_DIR, 'cov_tr.txt'))
    X_te = np.loadtxt(os.path.join(OUTPUT_DIR, 'cov_te.txt'))
    X_tr = np.hstack([X_tr, np.ones((X_tr.shape[0], 1))])
    X_te = np.hstack([X_te, np.ones((X_te.shape[0], 1))])
    np.savetxt(os.path.join(OUTPUT_DIR, 'cov_int_tr.txt'), X_tr)
    np.savetxt(os.path.join(OUTPUT_DIR, 'cov_int_te.txt'), X_te)

    # Append B-spline columns (age = column 0)
    Phi_tr = np.vstack([B(age) for age in X_tr[:, 0]])
    Phi_te = np.vstack([B(age) for age in X_te[:, 0]])
    X_tr   = np.hstack([X_tr, Phi_tr])
    X_te   = np.hstack([X_te, Phi_te])

    cov_tr_path = os.path.join(roi_dir, 'cov_bspline_tr.txt')
    cov_te_path = os.path.join(roi_dir, 'cov_bspline_te.txt')
    np.savetxt(cov_tr_path, X_tr)
    np.savetxt(cov_te_path, X_te)

    resp_tr_path = os.path.join(OUTPUT_DIR, f'resp_tr_{roi}.txt')
    resp_te_path = os.path.join(OUTPUT_DIR, f'resp_te_{roi}.txt')

    os.chdir(roi_dir)  # PCNToolkit writes blr/ subdirectory relative to cwd

    try:
        yhat_te, s2_te, nm, Z, metrics_te = estimate(
            cov_tr_path, resp_tr_path,
            testresp=resp_te_path, testcov=cov_te_path,
            alg='blr', optimizer='powell',
            cv_folds=5,
            savemodel=True, saveoutput=False,
            standardize=False,
        )

        np.savetxt(f'{roi}_yhat_te.txt', yhat_te)
        np.savetxt(f'{roi}_s2_te.txt',   s2_te)
        np.savetxt(f'{roi}_Z.txt',        Z)

        metrics_yhat_s2_Z.loc[len(metrics_yhat_s2_Z)] = [
            roi,
            yhat_te.flatten().tolist(),
            s2_te.flatten().tolist(),
            Z.flatten().tolist(),
        ]

        blr_metrics.loc[len(blr_metrics)] = [
            roi,
            metrics_te['MSLL'][0], metrics_te['EXPV'][0],
            metrics_te['SMSE'][0], metrics_te['RMSE'][0],
            metrics_te['Rho'][0],  metrics_te['pRho'][0],
        ]

        # Global evaluation
        y_te_arr = np.loadtxt(resp_te_path)[:, None]
        y_tr_arr = np.loadtxt(resp_tr_path)[:, None]
        mg = evaluate(
            y_te_arr, yhat_te, s2_te,
            np.array([y_tr_arr.mean()]), np.array([y_tr_arr.std()]),
        )
        blr_globmetrics.loc[len(blr_globmetrics)] = [
            roi, 'global',
            mg['MSLL'][0], mg['EXPV'][0],
            mg['SMSE'][0], mg['RMSE'][0],
            mg['Rho'][0],  mg['pRho'][0],
        ]

        # Site-level evaluation
        for num, site_idx in enumerate(sites):
            if len(site_idx) < MIN_SITE_N:
                continue
            ms = evaluate(
                y_te_arr[site_idx], yhat_te[site_idx], s2_te[site_idx],
                np.array([y_tr_arr.mean()]), np.array([y_tr_arr.std()]),
            )
            blr_site_metrics.loc[len(blr_site_metrics)] = [
                roi, site_names[num],
                ms['MSLL'][0], ms['EXPV'][0],
                ms['SMSE'][0], ms['RMSE'][0],
                ms['Rho'][0],  ms['pRho'][0],
            ]

    except Exception as e:
        print(f'  ERROR {roi}: {e}')

os.chdir(OUTPUT_DIR)

Running ROI: mr_y_smri__vol__aseg__ab__lh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__aseg__ab__lh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 30807.549351
         Iterations: 3
         Function evaluations: 89
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__aseg__ab__rh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__aseg__ab__rh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 30361.851059
         Iterations: 3
         Function e

/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=1.90303e-19): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)


Optimization terminated successfully.
         Current function value: 44281.879136
         Iterations: 3
         Function evaluations: 91
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__cn__lh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__cn__lh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 40022.787391
         Iterations: 3
         Function evaluations: 91
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__er__lh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__er__lh_sum.txt
Estimating model  1 o

/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.7247e-21): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)
/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.39321e-19): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)


Optimization terminated successfully.
         Current function value: 38578.532181
         Iterations: 3
         Function evaluations: 93
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__pcg__lh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__pcg__lh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 39683.599757
         Iterations: 3
         Function evaluations: 93
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__pfrt__lh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__pfrt__lh_sum.txt
Estimating mode

/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.7247e-21): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)
/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.82764e-19): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)


Optimization terminated successfully.
         Current function value: 39558.127968
         Iterations: 3
         Function evaluations: 95
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__cmfrt__rh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__cmfrt__rh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 44332.167311
         Iterations: 3
         Function evaluations: 92
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__cn__rh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__cn__rh_sum.txt
Estimating mode

/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.7247e-21): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)
/Users/nioushad/miniconda3/envs/normmodel310/lib/python3.10/site-packages/pcntoolkit/model/bayesreg.py:196: LinAlgWarning: Ill-conditioned matrix (rcond=7.86639e-19): result may not be accurate.
  invAXt = linalg.solve(self.A, X.T, check_finite=False)


Optimization terminated successfully.
         Current function value: 38655.493428
         Iterations: 3
         Function evaluations: 94
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__rmfrt__rh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__rmfrt__rh_sum.txt
Estimating model  1 of 1
configuring BLR ( order 1 )
Using default hyperparameters
Optimization terminated successfully.
         Current function value: 47467.771318
         Iterations: 3
         Function evaluations: 95
Saving model meta-data...
Evaluating the model ...
Running ROI: mr_y_smri__vol__dsk__sfrt__rh_sum
inscaler: None
outscaler: None
Processing data in /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/resp_tr_mr_y_smri__vol__dsk__sfrt__rh_sum.txt
Estimating 

## Step 7 — Save aggregated results

In [9]:
blr_metrics.to_csv(      os.path.join(OUTPUT_DIR, 'blr_metrics.csv'),       index=False)
blr_globmetrics.to_csv(  os.path.join(OUTPUT_DIR, 'blr_globmetrics.csv'),   index=False)
blr_site_metrics.to_csv( os.path.join(OUTPUT_DIR, 'blr_site_metrics.csv'),  index=False)
metrics_yhat_s2_Z.to_csv(os.path.join(OUTPUT_DIR, 'blr_yhat_s2_Z.csv'),     index=False)

print('Saved:')
for fname in ['blr_metrics.csv','blr_globmetrics.csv','blr_site_metrics.csv','blr_yhat_s2_Z.csv']:
    print(f'  {os.path.join(OUTPUT_DIR, fname)}')

Saved:
  /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/blr_metrics.csv
  /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/blr_globmetrics.csv
  /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/blr_site_metrics.csv
  /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/blr_yhat_s2_Z.csv


## Step 8 — ROI name mapping & summary stats

In [10]:
VOL_MAPPING = {
    # Subcortical
    'smri_vol_scs_tplh':     'THALAMUS_LEFT',      'smri_vol_scs_tprh':     'THALAMUS_RIGHT',
    'smri_vol_scs_caudatelh':'CAUDATE_LEFT',        'smri_vol_scs_caudaterh':'CAUDATE_RIGHT',
    'smri_vol_scs_putamenlh':'PUTAMEN_LEFT',        'smri_vol_scs_putamenrh':'PUTAMEN_RIGHT',
    'smri_vol_scs_pallidumlh':'PALLIDUM_LEFT',      'smri_vol_scs_pallidumrh':'PALLIDUM_RIGHT',
    'smri_vol_scs_hpuslh':   'HIPPOCAMPUS_LEFT',   'smri_vol_scs_hpusrh':   'HIPPOCAMPUS_RIGHT',
    'smri_vol_scs_amygdalalh':'AMYGDALA_LEFT',      'smri_vol_scs_amygdalarh':'AMYGDALA_RIGHT',
    'smri_vol_scs_aal':      'ACCUMBENS_LEFT',      'smri_vol_scs_aar':      'ACCUMBENS_RIGHT',
    # Cortical — Left
    'smri_vol_cdk_banksstslh':'L.bankssts',         'smri_vol_cdk_cdacatelh':'L.caudalanteriorcingulate',
    'smri_vol_cdk_cdmdfrlh': 'L.caudalmiddlefrontal','smri_vol_cdk_cuneuslh':'L.cuneus',
    'smri_vol_cdk_ehinallh': 'L.entorhinal',        'smri_vol_cdk_fusiformlh':'L.fusiform',
    'smri_vol_cdk_ifpllh':   'L.inferiorparietal',  'smri_vol_cdk_iftmlh':   'L.inferiortemporal',
    'smri_vol_cdk_ihcatelh': 'L.isthmuscingulate',  'smri_vol_cdk_locclh':   'L.lateraloccipital',
    'smri_vol_cdk_lobfrlh':  'L.lateralorbitofrontal','smri_vol_cdk_linguallh':'L.lingual',
    'smri_vol_cdk_mobfrlh':  'L.medialorbitofrontal','smri_vol_cdk_mdtmlh':   'L.middletemporal',
    'smri_vol_cdk_parahpallh':'L.parahippocampal',  'smri_vol_cdk_paracnlh': 'L.paracentral',
    'smri_vol_cdk_parsopclh':'L.parsopercularis',   'smri_vol_cdk_parsobislh':'L.parsorbitalis',
    'smri_vol_cdk_parstgrislh':'L.parstriangularis','smri_vol_cdk_pericclh': 'L.pericalcarine',
    'smri_vol_cdk_postcnlh': 'L.postcentral',       'smri_vol_cdk_ptcatelh': 'L.posteriorcingulate',
    'smri_vol_cdk_precnlh':  'L.precentral',        'smri_vol_cdk_pclh':     'L.precuneus',
    'smri_vol_cdk_rracatelh':'L.rostralanteriorcingulate','smri_vol_cdk_rrmdfrlh':'L.rostralmiddlefrontal',
    'smri_vol_cdk_sufrlh':   'L.superiorfrontal',   'smri_vol_cdk_supllh':   'L.superiorparietal',
    'smri_vol_cdk_sutmlh':   'L.superiortemporal',  'smri_vol_cdk_smlh':     'L.supramarginal',
    'smri_vol_cdk_frpolelh': 'L.frontalpole',       'smri_vol_cdk_tmpolelh': 'L.temporalpole',
    'smri_vol_cdk_trvtmlh':  'L.transversetemporal','smri_vol_cdk_insulalh': 'L.insula',
    # Cortical — Right
    'smri_vol_cdk_banksstsrh':'R.bankssts',         'smri_vol_cdk_cdacaterh':'R.caudalanteriorcingulate',
    'smri_vol_cdk_cdmdfrrh': 'R.caudalmiddlefrontal','smri_vol_cdk_cuneusrh':'R.cuneus',
    'smri_vol_cdk_ehinalrh': 'R.entorhinal',        'smri_vol_cdk_fusiformrh':'R.fusiform',
    'smri_vol_cdk_ifplrh':   'R.inferiorparietal',  'smri_vol_cdk_iftmrh':   'R.inferiortemporal',
    'smri_vol_cdk_ihcaterh': 'R.isthmuscingulate',  'smri_vol_cdk_loccrh':   'R.lateraloccipital',
    'smri_vol_cdk_lobfrrh':  'R.lateralorbitofrontal','smri_vol_cdk_lingualrh':'R.lingual',
    'smri_vol_cdk_mobfrrh':  'R.medialorbitofrontal','smri_vol_cdk_mdtmrh':   'R.middletemporal',
    'smri_vol_cdk_parahpalrh':'R.parahippocampal',  'smri_vol_cdk_paracnrh': 'R.paracentral',
    'smri_vol_cdk_parsopcrh':'R.parsopercularis',   'smri_vol_cdk_parsobisrh':'R.parsorbitalis',
    'smri_vol_cdk_parstgrisrh':'R.parstriangularis','smri_vol_cdk_periccrh': 'R.pericalcarine',
    'smri_vol_cdk_postcnrh': 'R.postcentral',       'smri_vol_cdk_ptcaterh': 'R.posteriorcingulate',
    'smri_vol_cdk_precnrh':  'R.precentral',        'smri_vol_cdk_pcrh':     'R.precuneus',
    'smri_vol_cdk_rracaterh':'R.rostralanteriorcingulate','smri_vol_cdk_rrmdfrrh':'R.rostralmiddlefrontal',
    'smri_vol_cdk_sufrrh':   'R.superiorfrontal',   'smri_vol_cdk_suplrh':   'R.superiorparietal',
    'smri_vol_cdk_sutmrh':   'R.superiortemporal',  'smri_vol_cdk_smrh':     'R.supramarginal',
    'smri_vol_cdk_frpolerh': 'R.frontalpole',       'smri_vol_cdk_tmpolerh': 'R.temporalpole',
    'smri_vol_cdk_trvtmrh':  'R.transversetemporal','smri_vol_cdk_insularh': 'R.insula',
}

vol_df = blr_globmetrics[blr_globmetrics['ROI'].str.contains('_vol_')].copy()
vol_df['ROI'] = vol_df['ROI'].replace(VOL_MAPPING)

print('Volume ROI global metrics — summary:')
print(vol_df[['MSLL','EXPV','Rho']].describe().round(3).to_string())

Volume ROI global metrics — summary:
         MSLL    EXPV     Rho
count  85.000  85.000  85.000
mean   -0.072   0.133   0.356
std     0.034   0.059   0.085
min    -0.164   0.022   0.152
25%    -0.093   0.089   0.299
50%    -0.074   0.137   0.373
75%    -0.046   0.171   0.415
max    -0.011   0.280   0.529


## Step 9 — Metric colorbars figure

In [11]:
COLORBARS = [
    ('summer', blr_globmetrics['MSLL'].min(), blr_globmetrics['MSLL'].max(), 'MSLL'),
    ('bwr',    blr_globmetrics['EXPV'].min(), blr_globmetrics['EXPV'].max(), 'EXPV'),
    ('RdPu',   blr_globmetrics['Rho'].min(),  blr_globmetrics['Rho'].max(),  'Rho'),
]

fig, axes = plt.subplots(1, 3, figsize=(10, 0.7))
for ax, (cmap_name, vmin, vmax, label) in zip(axes, COLORBARS):
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sm   = cm.ScalarMappable(norm=norm, cmap=cm.get_cmap(cmap_name))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=ax, orientation='horizontal')
    cbar.set_label(label, fontsize=10, labelpad=3)
    cbar.ax.tick_params(labelsize=8, pad=2)

plt.tight_layout(pad=0.5)
fig_path = os.path.join(OUTPUT_DIR, 'figures', 'metric_colorbars.svg')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.close()

print('Pipeline complete.')

Saved: /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/processt1/figures/metric_colorbars.svg
Pipeline complete.


/var/folders/cq/3591n_md2gq85yx6l4rgvk9m0000gn/T/ipykernel_58125/841737313.py:10: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  sm   = cm.ScalarMappable(norm=norm, cmap=cm.get_cmap(cmap_name))


In [13]:
blr_globmetrics['MSLL'].min()

-0.16387764638975333